# Processed USBL — Lonboard Visualisation

Batch-visualises a list of processed USBL CSV files (TrackLink or Evologics).
For each file: an interactive map of target fixes and ship track, followed by
time series of the target position in the vessel body frame.

In [ ]:
from pathlib import Path

import geopandas as gpd
import lonboard
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from shapely.geometry import LineString

In [ ]:
PROCESSED_DIR = Path("/data/exos_01/acfr_messages_v4_telemetry_processed")

FILES = [
    # TrackLink
    PROCESSED_DIR / "qdch0ftq_20100428_020202_usbl_tracklink.csv",
    PROCESSED_DIR / "qdch0ftq_20110415_020103_usbl_tracklink.csv",
    PROCESSED_DIR / "qdch0ftq_20120430_002423_usbl_tracklink.csv",
    PROCESSED_DIR / "qdch0ftq_20130406_023610_usbl_tracklink.csv",
    PROCESSED_DIR / "qdchdmy1_20110416_005411_usbl_tracklink.csv",
    PROCESSED_DIR / "qdchdmy1_20120501_071203_usbl_tracklink.csv",
    PROCESSED_DIR / "qdchdmy1_20130406_081713_usbl_tracklink.csv",
    # Evologics
    # PROCESSED_DIR / "qdch0ftq_20170526_025746_usbl_evologics.csv",
    # PROCESSED_DIR / "qdch0ftq_20210315_034028_usbl_evologics.csv",
    # PROCESSED_DIR / "qdchdmy1_20170525_234624_usbl_evologics.csv",
    # PROCESSED_DIR / "qdchdmy1_20210315_081519_usbl_evologics.csv",
    # PROCESSED_DIR / "qd61g27j_20170523_040815_usbl_evologics.csv",
    # PROCESSED_DIR / "qdc5ghs3_20210315_230947_usbl_evologics.csv",
    # PROCESSED_DIR / "qtqxshxs_20150327_015552_usbl_evologics.csv",
    # PROCESSED_DIR / "qtqxshxs_20150328_000850_usbl_evologics.csv",
    # PROCESSED_DIR / "qtqxshxs_20150328_042551_usbl_evologics.csv",
]

In [ ]:
# Target fixes: red-orange scatter
_TARGET_COLOR = [255, 80, 50, 220]
# Ship track: blue path
_SHIP_COLOR = [50, 130, 255, 200]


def make_map(df: pd.DataFrame) -> lonboard.Map:
    target_gdf = gpd.GeoDataFrame(
        geometry=gpd.points_from_xy(df["target_longitude"], df["target_latitude"]),
        crs="EPSG:4326",
    )
    target_layer = lonboard.ScatterplotLayer.from_geopandas(
        target_gdf,
        radius_min_pixels=6,
        get_fill_color=_TARGET_COLOR,
    )

    ship_coords = list(zip(df["ship_longitude"], df["ship_latitude"]))
    layers: list = [target_layer]
    if len(ship_coords) >= 2:
        ship_gdf = gpd.GeoDataFrame(
            geometry=[LineString(ship_coords)],
            crs="EPSG:4326",
        )
        ship_layer = lonboard.PathLayer.from_geopandas(
            ship_gdf,
            width_min_pixels=2,
            get_color=_SHIP_COLOR,
        )
        layers.insert(0, ship_layer)

    return lonboard.Map(layers=layers)


def make_timeseries(df: pd.DataFrame, title: str) -> None:
    timestamps = pd.to_datetime(df["timestamp"], format="ISO8601", utc=True)
    t = (timestamps - timestamps.iloc[0]).dt.total_seconds()

    ex = df.iloc[0]
    extrinsics_str = (
        f"extrinsics:  "
        f"x={ex['usbl_extrinsics_locx']:.3f} m  "
        f"y={ex['usbl_extrinsics_locy']:.3f} m  "
        f"z={ex['usbl_extrinsics_locz']:.3f} m  |  "
        f"φ={ex['usbl_extrinsics_rotx']:.4f} rad  "
        f"θ={ex['usbl_extrinsics_roty']:.4f} rad  "
        f"ψ={ex['usbl_extrinsics_rotz']:.4f} rad"
    )

    fig, axes = plt.subplot_mosaic(
        [["xs", "xv"], ["ys", "yv"], ["zs", "zv"], ["d", "d"]],
        figsize=(14, 10),
        sharex=True,
    )
    fig.suptitle(f"{title}\n{extrinsics_str}", fontsize=10)

    sensor_cols = [
        ("xs", "target_x_sensor", "target_x_sensor (m)"),
        ("ys", "target_y_sensor", "target_y_sensor (m)"),
        ("zs", "target_z_sensor", "target_z_sensor (m)"),
    ]
    vessel_cols = [
        ("xv", "target_x_vessel", "target_x_vessel (m, fwd)"),
        ("yv", "target_y_vessel", "target_y_vessel (m, stbd)"),
        ("zv", "target_z_vessel", "target_z_vessel (m, down)"),
    ]

    for key, col, label in sensor_cols:
        axes[key].scatter(t, df[col], s=10, color="C0")
        axes[key].axhline(0, color="k", lw=0.5, ls="--")
        axes[key].set_ylabel(label)
        axes[key].grid(True, alpha=0.3)

    for key, col, label in vessel_cols:
        axes[key].scatter(t, df[col], s=10, color="C1")
        axes[key].axhline(0, color="k", lw=0.5, ls="--")
        axes[key].set_ylabel(label)
        axes[key].grid(True, alpha=0.3)

    axes["d"].scatter(t, df["target_depth"], s=10, color="C2")
    axes["d"].set_ylabel("target_depth (m)")
    axes["d"].set_xlabel("time (s)")
    axes["d"].grid(True, alpha=0.3)

    axes["xs"].set_title("Sensor frame", fontsize=9)
    axes["xv"].set_title("Vessel frame", fontsize=9)

    fig.tight_layout()
    plt.show()

In [ ]:
for path in FILES:
    df = pd.read_csv(path)
    name = path.stem
    print(f"\n{'─' * 60}")
    print(f"{name}  ({len(df)} pings)")
    print(f"{'─' * 60}")
    display(make_map(df))
    make_timeseries(df, name)